# Transformer and supervised fine-tuning from first principles

This notebook builds a tiny causal language model and fine-tunes it on prompt/response pairs. It uses PyTorch tensors and autograd, but implements the Transformer attention calculation, causal mask, SFT label mask, loss, and generation loop directly.

## Learning objectives

By the end, you should be able to explain:

1. How token embeddings become contextual representations through causal self-attention.
2. Why a decoder-only model predicts one next token at every sequence position.
3. Why SFT is ordinary next-token cross-entropy applied to a curated dataset, usually with prompt-token losses masked out.
4. How teacher forcing during training differs from autoregressive generation at inference time.
5. What this toy example omits relative to real post-training.

## 1. The algorithm in one picture

For tokens $x_0, \ldots, x_{T-1}$, a causal language model estimates

$$p_\theta(x_1, \ldots, x_T) = \prod_{t=0}^{T-1} p_\theta(x_{t+1} \mid x_{\le t}).$$

A decoder-only Transformer repeatedly applies:

```text
token + position embeddings
        |
LayerNorm -> causal multi-head self-attention -> residual add
        |
LayerNorm -> position-wise MLP              -> residual add
        |
vocabulary logits for the next token at every position
```

SFT does not require a different architecture. It changes the examples and decides which next-token predictions contribute to the loss.

In [ ]:
from __future__ import annotations

import math
import random

import torch
from torch import nn
from torch.nn import functional as F

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

print(f"PyTorch {torch.__version__}; device=cpu")

## 2. Turn supervised examples into language-model tokens

Real systems use a subword tokenizer and a chat template. A character vocabulary keeps this example transparent. Every record is serialized as:

```text
<bos> prompt <sep> response <eos>
```

The input is the sequence without its last token. The target is the same sequence shifted one token left. Prompt targets become `IGNORE_INDEX`, so only response tokens and `<eos>` contribute to SFT loss. This shift-and-mask order is a common source of off-by-one bugs.

In [ ]:
pairs = [
    ("question: 2+2?", "answer: 4"),
    ("question: 3+4?", "answer: 7"),
    ("opposite of hot?", "cold"),
    ("opposite of up?", "down"),
    ("say hello", "hello"),
    ("say goodbye", "goodbye"),
]

special_tokens = ["<pad>", "<bos>", "<sep>", "<eos>"]
characters = sorted(set("".join(prompt + response for prompt, response in pairs)))
vocabulary = special_tokens + characters
stoi = {token: index for index, token in enumerate(vocabulary)}
itos = {index: token for token, index in stoi.items()}
PAD, BOS, SEP, EOS = (stoi[token] for token in special_tokens)
IGNORE_INDEX = -100

def encode_text(text: str) -> list[int]:
    return [stoi[character] for character in text]

def readable(token_ids: list[int]) -> str:
    return " ".join(itos[token_id] for token_id in token_ids)

def make_training_example(prompt: str, response: str):
    sequence = [BOS] + encode_text(prompt) + [SEP] + encode_text(response) + [EOS]
    # A role flag follows each sequence token: False for prompt/template, True for response.
    response_role = (
        [False] * (1 + len(prompt) + 1)
        + [True] * (len(response) + 1)
    )
    inputs = sequence[:-1]
    full_targets = sequence[1:]
    response_targets = [
        target if is_response else IGNORE_INDEX
        for target, is_response in zip(full_targets, response_role[1:])
    ]
    return inputs, full_targets, response_targets

examples = [make_training_example(*pair) for pair in pairs]
max_length = max(len(inputs) for inputs, _, _ in examples)

def collate(examples):
    batch_size = len(examples)
    input_ids = torch.full((batch_size, max_length), PAD, dtype=torch.long)
    full_labels = torch.full((batch_size, max_length), IGNORE_INDEX, dtype=torch.long)
    sft_labels = torch.full((batch_size, max_length), IGNORE_INDEX, dtype=torch.long)
    for row, (inputs, full_targets, response_targets) in enumerate(examples):
        length = len(inputs)
        input_ids[row, :length] = torch.tensor(inputs)
        full_labels[row, :length] = torch.tensor(full_targets)
        sft_labels[row, :length] = torch.tensor(response_targets)
    return input_ids, full_labels, sft_labels

input_ids, full_labels, sft_labels = collate(examples)
attention_mask = input_ids.ne(PAD)

first_length = attention_mask[0].sum().item()
print("input:     ", readable(input_ids[0, :first_length].tolist()))
print("next token:", readable(full_labels[0, :first_length].tolist()))
print("SFT target:", " ".join(
    "---" if token_id == IGNORE_INDEX else itos[token_id]
    for token_id in sft_labels[0, :first_length].tolist()
))

## 3. Implement causal multi-head self-attention

For each head, learned projections produce queries, keys, and values:

$$Q=XW_Q, \quad K=XW_K, \quad V=XW_V.$$

Attention weights are

$$A = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}} + M\right), \qquad \operatorname{Attention}(X)=AV,$$

where the causal mask $M_{ij}=-\infty$ when key position $j$ is in the future of query position $i$. Multiple heads run this operation in parallel and concatenate their results.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, max_seq_len: int):
        super().__init__()
        assert model_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.qkv = nn.Linear(model_dim, 3 * model_dim, bias=False)
        self.output = nn.Linear(model_dim, model_dim, bias=False)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x: torch.Tensor, token_mask: torch.Tensor):
        batch_size, seq_len, model_dim = x.shape
        qkv = self.qkv(x)
        qkv = qkv.view(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.causal_mask[:seq_len, :seq_len], float("-inf"))
        scores = scores.masked_fill(~token_mask[:, None, None, :], float("-inf"))
        weights = F.softmax(scores, dim=-1)

        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, model_dim)
        return self.output(context), weights


class TransformerBlock(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, max_seq_len: int):
        super().__init__()
        self.attention_norm = nn.LayerNorm(model_dim)
        self.attention = CausalSelfAttention(model_dim, num_heads, max_seq_len)
        self.mlp_norm = nn.LayerNorm(model_dim)
        self.mlp = nn.Sequential(
            nn.Linear(model_dim, 4 * model_dim),
            nn.GELU(),
            nn.Linear(4 * model_dim, model_dim),
        )

    def forward(self, x: torch.Tensor, token_mask: torch.Tensor):
        attended, weights = self.attention(self.attention_norm(x), token_mask)
        x = x + attended
        x = x + self.mlp(self.mlp_norm(x))
        return x, weights


class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, model_dim=48, num_heads=4, layers=2):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, model_dim)
        self.position_embedding = nn.Embedding(max_seq_len, model_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(model_dim, num_heads, max_seq_len) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(model_dim)
        self.lm_head = nn.Linear(model_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # common weight tying

    def forward(self, token_ids: torch.Tensor, token_mask: torch.Tensor | None = None):
        if token_mask is None:
            token_mask = token_ids.ne(PAD)
        positions = torch.arange(token_ids.shape[1], device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        attention_maps = []
        for block in self.blocks:
            x, weights = block(x, token_mask)
            attention_maps.append(weights)
        return self.lm_head(self.final_norm(x)), attention_maps


model = TinyCausalLM(len(vocabulary), max_length)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Vocabulary: {len(vocabulary)} tokens; model: {parameter_count:,} parameters")

In [ ]:
# Verify the defining causal invariant: no head can attend to a future position.
with torch.no_grad():
    _, attention_maps = model(input_ids, attention_mask)
first_head = attention_maps[0][0, 0]
future_attention = torch.triu(first_head, diagonal=1).abs().max().item()
print(f"Maximum attention weight assigned to the future: {future_attention:.1f}")
assert future_attention == 0.0

query_position = first_length - 1
valid_weights = first_head[query_position, :first_length]
top_weights, top_positions = valid_weights.topk(min(5, first_length))
print(f"At the last input token ({itos[input_ids[0, query_position].item()]}), head 0 attends most to:")
for weight, position in zip(top_weights.tolist(), top_positions.tolist()):
    print(f"  position {position:2d} token={itos[input_ids[0, position].item()]!r:8s} weight={weight:.3f}")

## 4. Define response-only SFT loss

Teacher forcing gives the model the ground-truth prefix in parallel and scores the correct next token. For a set $R$ of response-token positions, the SFT objective is

$$\mathcal{L}_{\mathrm{SFT}}(\theta) = -\frac{1}{|R|}\sum_{t \in R} \log p_\theta(x_{t+1} \mid x_{\le t}).$$

Framework cross-entropy can implement this directly, but spelling it out makes the mask semantics visible.

In [ ]:
def masked_next_token_loss(logits: torch.Tensor, labels: torch.Tensor):
    selected = labels.ne(IGNORE_INDEX)
    log_probabilities = F.log_softmax(logits, dim=-1)
    row, position = selected.nonzero(as_tuple=True)
    correct_token = labels[row, position]
    token_losses = -log_probabilities[row, position, correct_token]
    return token_losses.mean(), token_losses

logits, _ = model(input_ids, attention_mask)
initial_sft_loss, token_losses = masked_next_token_loss(logits, sft_labels)
print(f"Response tokens scored: {token_losses.numel()}")
print(f"Initial mean SFT loss: {initial_sft_loss.item():.3f}")

## 5. Teacher-forced training and autoregressive generation

Training computes every position in parallel because the causal mask prevents leakage. Generation cannot do that: it samples or selects one new token, appends it, and runs the model again. Production systems cache keys and values so earlier positions are not recomputed.

In [ ]:
@torch.no_grad()
def generate(model: TinyCausalLM, prompt: str, max_new_tokens=20) -> str:
    model.eval()
    generated = [BOS] + encode_text(prompt) + [SEP]
    response = []
    for _ in range(max_new_tokens):
        context = torch.tensor([generated[-model.max_seq_len:]], dtype=torch.long)
        logits, _ = model(context, torch.ones_like(context, dtype=torch.bool))
        next_token = logits[0, -1].argmax().item()
        if next_token == EOS:
            break
        generated.append(next_token)
        if next_token not in (PAD, BOS, SEP):
            response.append(itos[next_token])
    return "".join(response)

print("Before SFT:")
for prompt, expected in pairs[:3]:
    print(f"  {prompt!r} -> {generate(model, prompt)!r} (target {expected!r})")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
model.train()

for step in range(301):
    optimizer.zero_grad(set_to_none=True)
    logits, _ = model(input_ids, attention_mask)
    loss, _ = masked_next_token_loss(logits, sft_labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    if step % 50 == 0:
        print(f"step={step:3d}  loss={loss.item():.4f}")

print("\nAfter SFT:")
for prompt, expected in pairs:
    print(f"  {prompt!r} -> {generate(model, prompt)!r} (target {expected!r})")

The tiny model can memorize six examples. This confirms that the data pipeline, causal model, loss mask, backward pass, and generation loop are connected correctly. It does **not** demonstrate generalization. Real SFT starts from a pretrained model whose representations already encode language and broad capabilities.

## 6. Prompt masking changes the optimization signal

A full-sequence language-model loss also trains the model to reproduce prompts and template text. Response-only SFT conditions on those tokens but does not reward predicting them. Both objectives backpropagate through prompt representations because response predictions attend to the prompt; masking a prompt label does **not** remove the prompt from the computation graph.

In [ ]:
# Use a fresh model: the trained model has nearly zero response loss and gradients.
torch.manual_seed(SEED + 1)
gradient_probe = TinyCausalLM(len(vocabulary), max_length)

def selected_gradient_norms(labels: torch.Tensor) -> dict[str, float]:
    gradient_probe.zero_grad(set_to_none=True)
    logits, _ = gradient_probe(input_ids, attention_mask)
    loss, _ = masked_next_token_loss(logits, labels)
    loss.backward()
    return {
        "token embedding / tied head": gradient_probe.token_embedding.weight.grad.norm().item(),
        "first QKV projection": gradient_probe.blocks[0].attention.qkv.weight.grad.norm().item(),
        "first MLP input": gradient_probe.blocks[0].mlp[0].weight.grad.norm().item(),
    }

full_norms = selected_gradient_norms(full_labels)
sft_norms = selected_gradient_norms(sft_labels)
print(f"{'parameter group':30s} {'full LM':>10s} {'response SFT':>14s}")
for name in full_norms:
    print(f"{name:30s} {full_norms[name]:10.4f} {sft_norms[name]:14.4f}")

## 7. Decoder-only LLM SFT algorithm, component by component

### Notation

- $\theta$: model parameters; $N=|\theta|$ is the parameter count.
- $B$: sequences in a minibatch; $S$: padded sequence length.
- $T$: token positions actually processed in the minibatch. A dense padded batch uses $T=BS$; efficient packing makes $T$ close to the number of useful tokens.
- $M$: supervised response-token positions ($M \le T$).
- $L$: Transformer layers; $H$: hidden width; $A$: attention heads.
- $V$: vocabulary size.

### Algorithm

```text
inputs: pretrained decoder-only LM parameters theta
        prompt/response records D
        chat template and tokenizer
        optimizer, learning-rate schedule, and stopping rule

repeat for each minibatch of records:
    1. serialize: apply the exact chat template to prompt and response roles
    2. tokenize: convert text to token IDs; append required special tokens
    3. batch: truncate or pack records, pad sequences, create padding mask
    4. label: shift token IDs left by one position to make next-token targets
    5. loss-mask: set prompt, padding, and unwanted-role labels to IGNORE_INDEX
    6. forward: run causal Transformer blocks and vocabulary projection
    7. objective: average next-token cross-entropy over non-ignored labels
    8. backward: use reverse-mode autodiff to compute every parameter gradient
    9. stabilize: optionally unscale mixed-precision gradients and clip their norm
   10. update: apply AdamW (or another optimizer) and advance the LR schedule
   11. clear gradients and continue

periodically evaluate held-out behavior and save model plus training state
output: updated parameters theta_SFT
```

### What each component does

| Component | Purpose | Common failure |
|---|---|---|
| Pretrained checkpoint | Supplies language, knowledge, and capabilities to shape | Training from random initialization tests mechanics but is not normal SFT |
| Curated records | Define the demonstrated behavior $p(\text{response}\mid\text{prompt})$ | Duplication, contradictions, contamination, or a poor task mixture |
| Chat template | Converts roles and turns into the token sequence seen by the LM | Training and inference templates disagree |
| Tokenizer | Maps strings and special markers to discrete token IDs | Missing EOS, incorrect role token, or unexpected truncation |
| Packing/padding mask | Forms efficient fixed-shape minibatches without cross-example attention | One example attends into another, or padding enters the loss |
| Causal mask | Prevents position $t$ from reading future tokens | Future leakage makes training loss invalid |
| Shifted labels | Align the logit at $t$ with target token $t+1$ | Off-by-one supervision |
| Response loss mask | Scores assistant tokens while retaining prompt tokens as context | Masking the first response token or supervising unwanted roles |
| Cross-entropy | Increases likelihood of demonstrated response tokens | Token-mean reduction overweights longer responses |
| Backpropagation | Applies the chain rule through logits, blocks, attention, and embeddings | Overflow, unstable norms, or excessive activation memory |
| AdamW and LR schedule | Turn gradients into controlled parameter updates | Learning rate causes forgetting or ineffective updates |
| Evaluation/checkpointing | Detects generalization, regressions, and recoverable progress | Judging success only from training loss |

Masking leaves only $M$ labels in the scalar loss, but it does **not** reduce the Transformer input to $M$ tokens. The model still processes the prompt, and response gradients flow backward through prompt representations via attention. Standard dense implementations therefore perform nearly all block computation over $T$ tokens.

### What matters in real SFT

- **Data quality and mixture:** examples define the behavior being cloned. Dataset composition can dominate optimizer details.
- **Template correctness:** training and inference must agree on roles, separators, BOS, and EOS.
- **Loss normalization:** token-mean and example-mean objectives weight the dataset differently.
- **Mask policy:** response-only, full-sequence, and multi-turn assistant-only masks produce different gradients.
- **Optimization scale:** learning rate, effective batch tokens, precision, sequence packing, and distributed reduction affect stability and throughput.
- **Evaluation:** low training loss may only show memorization. Measure held-out tasks, regressions, formatting, safety, and calibration.

## 8. Compute and memory/storage accounting

The expressions $2NT$ and $4NT$ describe approximate **floating-point work**, not storage. Here $N$ is the number of dense model parameters and $T$ is the number of token positions processed in one optimizer step. A multiply-add is counted as two FLOPs.

### Compute per training step

For a dense linear weight matrix, the forward matrix multiplication costs about two FLOPs per weight per token. Summed over the model's dense weights:

$$C_{\text{forward}} \approx 2NT.$$

During backward, a linear layer computes both a weight gradient and an input gradient. Each is approximately one forward matmul:

$$C_{\text{backward}} \approx 4NT, \qquad C_{\text{train}} \approx 6NT.$$

These are useful scaling rules, not exact counts. They omit embeddings, normalization, activations, loss, optimizer work, communication, and attention's explicit score/value products. Standard attention adds roughly

$$4BLS^2H$$

forward FLOPs for $QK^\top$ and $AV$ together. The dense $2NT$ rule is most accurate when parameterized projections and MLPs dominate; the quadratic attention term matters at long context. Counting a fused multiply-add as one operation divides the quoted FLOPs by two but does not change their ratios.

### Persistent parameter and optimizer memory

If a value uses $b$ bytes, $N$ such values require $bN$ bytes. Typical per-parameter accounting is:

| Training state | Bytes per parameter | Why |
|---|---:|---|
| BF16/FP16 inference weights | 2 | One low-precision model weight |
| FP32 inference weights | 4 | One full-precision model weight |
| FP32 AdamW training state | 16 | weight 4 + gradient 4 + first moment 4 + second moment 4 |
| Common mixed-precision AdamW | about 16 | low-precision weight 2 + gradient 2 + FP32 master weight 4 + moments 8 |

Implementations vary: gradients may be FP32, master weights may be omitted, optimizer states may be quantized, and distributed sharding may divide these tensors across workers. Thus `16N bytes` is a planning baseline, not a law. A weights-only BF16 checkpoint is about $2N$ bytes; a resumable training checkpoint also stores optimizer state, scheduler/scaler state, and RNG state.

### Runtime activation memory

Training must retain or recompute intermediate tensors for backward:

- Hidden-state activations scale approximately as $O(LTH)$.
- Materialized attention probabilities scale as $O(BLAS^2)$. Flash-style attention avoids storing the full $S\times S$ matrix.
- Vocabulary logits can scale as $O(BSV)$ if fully materialized. Fused or chunked cross-entropy reduces this peak.
- Gradient checkpointing stores fewer layer activations and recomputes them during backward, trading memory for FLOPs.

Peak training memory is therefore not just `16N`: it is parameter/optimizer state plus activations, temporary workspaces, communication buffers, and framework overhead. Sequence length and microbatch size control the activation part.

### Autoregressive inference storage

Generation stores a KV cache so previous keys and values are not recomputed. Its approximate size is

$$M_{\text{KV}} \approx 2LBSH_{\text{KV}}b \text{ bytes},$$

where the leading 2 means key plus value, $H_{\text{KV}}$ is the combined width of KV heads, and $b$ is bytes per cache element. For ordinary multi-head attention $H_{\text{KV}}=H$; multi-query or grouped-query attention reduces it. The KV cache is an inference concern, not normally the main SFT-training store.

### Concrete 7B example

For $N=7$ billion parameters and $T=8192$ tokens in one step:

- BF16 weights only: about 14 GB (13.0 GiB).
- Unsharded AdamW baseline at 16 bytes/parameter: about 112 GB (104.3 GiB), before activations.
- Forward dense work: about 115 trillion FLOPs.
- Backward dense work: about 229 trillion FLOPs.
- Total dense training work: about 344 trillion FLOPs, plus omitted terms.

In [ ]:
def format_bytes(byte_count: int) -> str:
    if byte_count >= 2**30:
        return f"{byte_count / 2**30:,.3f} GiB"
    return f"{byte_count / 2**20:,.3f} MiB"

def format_flops(flops: int) -> str:
    if flops >= 10**12:
        return f"{flops / 1e12:,.3f} TFLOPs"
    if flops >= 10**9:
        return f"{flops / 1e9:,.3f} GFLOPs"
    return f"{flops / 1e6:,.3f} MFLOPs"

def show_cost_estimate(name: str, parameters: int, tokens: int) -> None:
    forward = 2 * parameters * tokens
    backward = 4 * parameters * tokens
    print(name)
    print(f"  parameters:              {parameters:,}")
    print(f"  tokens this step:         {tokens:,}")
    print(f"  BF16 weights:             {format_bytes(2 * parameters)}")
    print(f"  AdamW baseline (16N):     {format_bytes(16 * parameters)} + activations")
    print(f"  forward dense compute:    {format_flops(forward)}")
    print(f"  backward dense compute:   {format_flops(backward)}")
    print(f"  total dense train compute:{format_flops(forward + backward)}")

tokens_in_toy_step = input_ids.numel()  # dense padded execution uses B * S positions
show_cost_estimate("Notebook model", parameter_count, tokens_in_toy_step)
show_cost_estimate("Illustrative 7B model", 7_000_000_000, 8_192)

## 9. Exercises

1. Replace `sft_labels` with `full_labels` during training. Compare prompt reproduction, response learning, and gradient norms.
2. Remove the causal mask and rerun the invariant check. Explain why the training loss becomes invalid even if it decreases faster.
3. Change from four attention heads to one while keeping model width fixed. Track parameter count and learning behavior.
4. Add one contradictory response. Observe which answer the model produces and why dataset counts matter.
5. Add a long response and compare token-mean loss with a per-example normalized loss.
6. Split examples into train and held-out prompts to make the memorization/generalization distinction measurable.
7. Add a second user/assistant turn and construct a role mask that supervises assistant turns only.